In [61]:
import pandas as pd
import numpy as np
import random
import pickle
import networkx as nx
from io import StringIO

from google.cloud import bigquery

import matplotlib.pyplot as plt
from matplotlib import ticker

from cluster_feature_functions import *
from growth_forecasting_feature_functions import *

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
import statsmodels.api as smf
import scipy.stats as stats
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from tqdm import tqdm

In [2]:
cset_colors = ["#0B1F41","#003DA6","#B53A6D","#7AC4A5","#F17F4C","#15AFD0","#839DC5","#E5BF21"]

In [3]:
client = bigquery.Client()

# Model Parameters

In [4]:
def zscore(column):
    zcolumn = (column - column.mean())/column.std()
    return zcolumn
def zfieldlog(column):
    column2 = column.replace(0,1e-7)
    column2 = np.log(column2)
    return zscore(column2)
def zlog(column):
    column2 = column.replace(0,1)
    column2 = np.log(column2)
    return zscore(column2)
def zfourth(column):
    column2 = column**0.25
    return zscore(column2)
def zcut(column):
    zcolumn = zscore(column)
    return zcolumn.clip(upper=4,lower=-4)
def znormed(column):
    column2 = zcut(column)
    return column2/4

In [42]:
transforms1 = {
    'L0' : zscore,
    'L1' : zlog,
    'L2' : zlog,
    'L3' : zscore,
    'L4' : zlog,
    'L5' : zscore,
    'L6' : zscore,
    'L7' : zscore,
    'L8' : zscore,
    'L9' : zscore,
    'L10': zscore,
    'L11': zscore,
    'L12': zscore,
    'L13': zscore,
    'L14': zscore,
    'L15': zscore,
    'I0' : zlog,
    'I1' : zlog,
    'D0' : zlog,
    'D1' : zlog,
    'D2' : zlog,
    'D3' : zlog,
    'F0' : zfieldlog,
    'F1' : zfieldlog,
    'F2' : zfieldlog,
    'F3' : zfieldlog,
    'F4' : zfieldlog,
    'F5' : zfieldlog,
    'F6' : zfieldlog,
    'F7' : zfieldlog,
    'F8' : zfieldlog,
    'F9' : zfieldlog,
    'F10': zfieldlog,
    'F11': zfieldlog,
    'N0' : zscore,
    'N1' : zscore,
    'N2' : zlog,
    'N3' : zlog,
    'N4' : zlog,
    'N5' : zlog,
    'N6' : zlog,
    'P0' : zscore,
    'CC0': zscore,
    'CC1': zscore,
    'CC2': zscore,
    'C0' : zscore,
    'C1' : zscore,
    'C2' : zscore,
    'S0': zscore,
    'S1': zscore,
    'S2': zscore,
    'S3': zscore,
    'S4': zscore,
    'S5': zscore,
    'S6': zscore,
    'S7': zscore,
    'S8': zscore,
    'S9': zscore,
    'S10': zscore,
    'S11': zscore,
    'S12': zscore,
    'S13': zscore,
    'S14': zscore,
    'S15': zscore,
    'dS0': zscore,
    'dS1': zscore,
    'dS2': zscore,
    'dS3': zscore,
    'dS4': zscore,
    'dS5': zscore,
    'dS6': zscore,
    'dS7': zscore,
    'dS8': zscore,
    'dS9': zscore,
    'dS10': zscore,
    'dS11': zscore,
    'dS12': zscore,
    'dS13': zscore,
    'dS14': zscore,
    'CS0': zscore,
    'CS1': zscore,
    'CS2': zscore,
    'CS3': zscore,
    'CS4': zscore,
    'CS5': zscore,
    'CS6': zscore,
    'CS7': zscore,
    'CS8': zscore,
    'CS9': zscore,
    'CS10': zscore,
    'CS11': zscore,
    'CS12': zscore,
    'CS13': zscore,
    'CS14': zscore,
    'CS15': zscore,
    'dCS0': zscore,
    'dCS1': zscore,
    'dCS2': zscore,
    'dCS3': zscore,
    'dCS4': zscore,
    'dCS5': zscore,
    'dCS6': zscore,
    'dCS7': zscore,
    'dCS8': zscore,
    'dCS9': zscore,
    'dCS10': zscore,
    'dCS11': zscore,
    'dCS12': zscore,
    'dCS13': zscore,
    'dCS14': zscore,
    'RS0': zscore,
    'RS1': zscore,
    'RS2': zscore,
    'RS3': zscore,
    'RS4': zscore,
    'RS5': zscore,
    'RS6': zscore,
    'RS7': zscore,
    'RS8': zscore,
    'RS9': zscore,
    'RS10': zscore,
    'RS11': zscore,
    'RS12': zscore,
    'RS13': zscore,
    'RS14': zscore,
    'RS15': zscore,
    'dRS0': zscore,
    'dRS1': zscore,
    'dRS2': zscore,
    'dRS3': zscore,
    'dRS4': zscore,
    'dRS5': zscore,
    'dRS6': zscore,
    'dRS7': zscore,
    'dRS8': zscore,
    'dRS9': zscore,
    'dRS10': zscore,
    'dRS11': zscore,
    'dRS12': zscore,
    'dRS13': zscore,
    'dRS14': zscore,
}
transforms3 = {
    'L0' : zscore,
    'L1' : zlog,
    'L2' : zlog,
    'L3' : zscore,
    'L4' : zlog,
    'L5' : zscore,
    'L6' : zscore,
    'L7' : zscore,
    'L8' : zscore,
    'L9' : zscore,
    'L10': zscore,
    'L11': zscore,
    'L12': zscore,
    'L13': zscore,
    'L14': zscore,
    'L15': zscore,
    'I0' : zlog,
    'I1' : zlog,
    'D0' : zlog,
    'D1' : zlog,
    'D2' : zlog,
    'D3' : zlog,
    'F0' : zfieldlog,
    'F1' : zfieldlog,
    'F2' : zfieldlog,
    'F3' : zfieldlog,
    'F4' : zfieldlog,
    'F5' : zfieldlog,
    'F6' : zfieldlog,
    'F7' : zfieldlog,
    'F8' : zfieldlog,
    'F9' : zfieldlog,
    'F10': zfieldlog,
    'F11': zfieldlog,
    'N0' : zscore,
    'N1' : zscore,
    'N2' : zlog,
    'N3' : zlog,
    'N4' : zlog,
    'N5' : zlog,
    'N6' : zlog,
    'P0' : zscore,
    'CC0': zscore,
    'CC1': zscore,
    'CC2': zscore,
    'C0' : zscore,
    'C1' : zscore,
    'C2' : zscore,
}
transforms_old = {
    'L0' : zscore,
    'L1' : zfourth,
    'L2' : zlog,
    'I0' : zlog,
}
transforms2 = {k: zcut for k in transforms3.keys()}
transforms_all = {k: zcut for k in transforms1.keys()}

In [9]:
# Loading in testing and training data
with open('training_data.pckl','rb') as fil:
    training_data = pickle.load(fil)
with open('testing_data.pckl','rb') as fil:
    testing_data = pickle.load(fil)

In [11]:
FY_max = 2022
growth_year = 'EG_3Yr'

train_set = [training_data[training_data['forecast_year'] == FY] for FY in range(2015, FY_max + 1,1)]
test_set = [testing_data[testing_data['forecast_year'] == FY] for FY in range(2015, FY_max + 1,1)]

x_test = pd.concat([pd.concat([transforms[k](x[k]) for k in transforms.keys()], axis=1) for x in test_set])
x_train = pd.concat([pd.concat([transforms[k](x[k]) for k in transforms.keys()], axis=1) for x in train_set])

y_test = pd.concat([x[[growth_year]] for x in test_set]).rename(columns={growth_year:'EG'})
y_train = pd.concat([x[[growth_year]] for x in train_set]).rename(columns={growth_year:'EG'})

print(f"Length of test data is: {len(x_test):,.0f}")
print(f"Length of train data is: {len(x_train):,.0f}")

# Oversampling testing data
os = RandomOverSampler(random_state=0)
os_datax, os_datay = os.fit_resample(x_train, y_train)

print(f"Length of oversampled training data is: {len(os_datax):,.0f}")
print(f"N extreme growth in oversampled training data is: {sum(os_datay['EG'] == 1):,.0f}")
print(f"N no extreme growth in oversampled training data is: {sum(os_datay['EG'] == 0):,.0f}")

Length of test data is: 146,544
Length of train data is: 586,208
Length of oversampled training data is: 1,082,070
N extreme growth in oversampled training data is: 541,035
N no extreme growth in oversampled training data is: 541,035


In [12]:
def probit_f(z):
    return stats.norm.cdf(z,0,1)

def extreme_growth_pred(z):
    pf = probit_f(z)
    return (pf > 0.5).astype(int)

In [13]:
# turning the training into a function for params of interes
def probit_model_fitting_eval(growth_year, FY_max, transforms):
    print("************************")
    print(f"Doing FY {FY_max} ({growth_year})")
    print("************************")
    # loading
    
    train_set = [training_data[training_data['forecast_year'] == FY] for FY in range(2015, FY_max + 1,1)]
    test_set = [testing_data[testing_data['forecast_year'] == FY] for FY in range(2015, FY_max + 1,1)]

    x_test = pd.concat([pd.concat([transforms[k](x[k]) for k in transforms.keys()], axis=1) for x in test_set])
    x_train = pd.concat([pd.concat([transforms[k](x[k]) for k in transforms.keys()], axis=1) for x in train_set])

    y_test = pd.concat([x[[growth_year]] for x in test_set]).rename(columns={growth_year:'EG'})
    y_train = pd.concat([x[[growth_year]] for x in train_set]).rename(columns={growth_year:'EG'})

    print(f"Length of test data is: {len(x_test):,.0f}")
    print(f"Length of train data is: {len(x_train):,.0f}")

    # Oversampling testing data
    os = RandomOverSampler(random_state=0)
    os_datax, os_datay = os.fit_resample(x_train, y_train)

    print(f"Length of oversampled training data is: {len(os_datax):,.0f}")
    print(f"N extreme growth in oversampled training data is: {sum(os_datay['EG'] == 1):,.0f}")
    print(f"N no extreme growth in oversampled training data is: {sum(os_datay['EG'] == 0):,.0f}")

    probit_model = smf.Probit(os_datay,os_datax)
    result = probit_model.fit()
    params = result.params
    y_pred_test = extreme_growth_pred(sum([x_test[column]*params[column] for column in params.index]))
    y_pred_train = extreme_growth_pred(sum([x_train[column]*params[column] for column in params.index]))

    eval_dict = {
        'growth_year'      : growth_year,
        'FY_max'           : FY_max,
        'accuracy_test'    : accuracy_score(y_test, y_pred_test),
        'precision_test'   : precision_score(y_test, y_pred_test),
        'recall_test'      : recall_score(y_test, y_pred_test),
        'f1_test'          : f1_score(y_test, y_pred_test),
        'accuracy_train'    : accuracy_score(y_train, y_pred_train),
        'precision_train'   : precision_score(y_train, y_pred_train),
        'recall_train'      : recall_score(y_train, y_pred_train),
        'f1_train'          : f1_score(y_train, y_pred_train),
        'params'           : params,
        'result table'     : result.summary().tables[1]
    }
    
    return(eval_dict)

In [47]:
simple_model_evals = []

transforms = transforms2
for growth_year, FY_max in zip(['EG_1Yr','EG_2Yr','EG_3Yr','EG_4Yr','EG_5Yr'],[2024,2023,2022,2021,2020]):
    simple_model_evals.append(probit_model_fitting_eval(growth_year, FY_max, transforms))


************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.385517
         Iterations 7
************************
Doing FY 2023 (EG_2Yr)
************************
Length of test data is: 164,862
Length of train data is: 659,484
Length of oversampled training data is: 1,231,340
N extreme growth in oversampled training data is: 615,670
N no extreme growth in oversampled training data is: 615,670
Optimization terminated successfully.
         Current function value: 0.374463
         Iterations 7
************************
Doing FY 2022 (EG_3Yr)
************************
Length of test data is: 146,544
Length of train data is: 586,208
Length of oversampled training data is: 1,08

In [48]:
full_model_evals = []

transforms = transforms1
for growth_year, FY_max in zip(['EG_1Yr','EG_2Yr','EG_3Yr','EG_4Yr','EG_5Yr'],[2024,2023,2022,2021,2020]):
    full_model_evals.append(probit_model_fitting_eval(growth_year, FY_max, transforms))


************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.413115
         Iterations 8
************************
Doing FY 2023 (EG_2Yr)
************************
Length of test data is: 164,862
Length of train data is: 659,484
Length of oversampled training data is: 1,231,340
N extreme growth in oversampled training data is: 615,670
N no extreme growth in oversampled training data is: 615,670
Optimization terminated successfully.
         Current function value: 0.401649
         Iterations 8
************************
Doing FY 2022 (EG_3Yr)
************************
Length of test data is: 146,544
Length of train data is: 586,208
Length of oversampled training data is: 1,08

In [38]:
current_model_evals = []

transforms = transforms_old
for growth_year, FY_max in zip(['EG_1Yr','EG_2Yr','EG_3Yr','EG_4Yr','EG_5Yr'],[2024,2023,2022,2021,2020]):
    current_model_evals.append(probit_model_fitting_eval(growth_year, FY_max, transforms))


************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.422442
         Iterations 6
************************
Doing FY 2023 (EG_2Yr)
************************
Length of test data is: 164,862
Length of train data is: 659,484
Length of oversampled training data is: 1,231,340
N extreme growth in oversampled training data is: 615,670
N no extreme growth in oversampled training data is: 615,670
Optimization terminated successfully.
         Current function value: 0.415631
         Iterations 6
************************
Doing FY 2022 (EG_3Yr)
************************
Length of test data is: 146,544
Length of train data is: 586,208
Length of oversampled training data is: 1,08

In [49]:
all_model_evals = []

transforms = transforms_all
for growth_year, FY_max in zip(['EG_1Yr','EG_2Yr','EG_3Yr','EG_4Yr','EG_5Yr'],[2024,2023,2022,2021,2020]):
    all_model_evals.append(probit_model_fitting_eval(growth_year, FY_max, transforms))

************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.381248
         Iterations 7
************************
Doing FY 2023 (EG_2Yr)
************************
Length of test data is: 164,862
Length of train data is: 659,484
Length of oversampled training data is: 1,231,340
N extreme growth in oversampled training data is: 615,670
N no extreme growth in oversampled training data is: 615,670
Optimization terminated successfully.
         Current function value: 0.369693
         Iterations 7
************************
Doing FY 2022 (EG_3Yr)
************************
Length of test data is: 146,544
Length of train data is: 586,208
Length of oversampled training data is: 1,08

In [140]:
pruned_model_evals = []

transforms = transforms_all
for growth_year, FY_max in zip(['EG_1Yr','EG_2Yr','EG_3Yr','EG_4Yr','EG_5Yr'],[2024,2023,2022,2021,2020]):
    
    transforms_tmp = {k: zcut for k in transforms_all}
    n_params = len(transforms_tmp)

    prune = True
    iteration = 1
    while prune:
        print(f"Running iteration {iteration} for pruning with {n_params} parameters")
        results_tmp = probit_model_fitting_eval(growth_year, FY_max, transforms_tmp)
        table_tmp = results_tmp['result table']
        df_tmp = pd.read_html(StringIO(table_tmp.as_html()))[0].set_index(0)
        df_tmp.columns = df_tmp.iloc[0]
        df_tmp = df_tmp[1:]
        pruned_params = df_tmp[df_tmp['P>|z|'].astype(float) <= 0.01].index
        transforms_tmp = {k: zcut for k in pruned_params}
        if len(transforms_tmp) < n_params:
            n_params = len(transforms_tmp)
            iteration += 1
            print(" ")
        else:
            prune = False
    
    pruned_model_evals.append(results_tmp)

Running iteration 1 for pruning with 141 parameters
************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.381248
         Iterations 7
 
Running iteration 2 for pruning with 90 parameters
************************
Doing FY 2024 (EG_1Yr)
************************
Length of test data is: 183,180
Length of train data is: 732,760
Length of oversampled training data is: 1,394,400
N extreme growth in oversampled training data is: 697,200
N no extreme growth in oversampled training data is: 697,200
Optimization terminated successfully.
         Current function value: 0.381462
         Iterations 7
 
Running iteration 3 for pruning with 87 parameters
************************

In [137]:
simple_model_evals = pd.DataFrame(simple_model_evals)
full_model_evals = pd.DataFrame(full_model_evals)
current_model_evals = pd.DataFrame(current_model_evals)
all_model_evals = pd.DataFrame(all_model_evals)
pruned_model_evals = pd.DataFrame(pruned_model_evals)
with open(f'ML_models/finalProbitModels/simple_model_evals_{len(transforms2)}features.pckl', 'wb') as fil:
    pickle.dump(simple_model_evals, fil)
with open(f'ML_models/finalProbitModels/full_model_evals_{len(transforms1)}features.pckl', 'wb') as fil:
    pickle.dump(full_model_evals, fil)
with open(f'ML_models/finalProbitModels/current_model_evals_{len(transforms_old)}features.pckl', 'wb') as fil:
    pickle.dump(current_model_evals, fil)
with open(f'ML_models/finalProbitModels/all_model_evals_{len(transforms_all)}features.pckl', 'wb') as fil:
    pickle.dump(all_model_evals, fil)
with open(f'ML_models/finalProbitModels/pruned_model_evals_variablefeatures.pckl', 'wb') as fil:
    pickle.dump(pruned_model_evals, fil)

In [138]:
full_model_evals[['growth_year','FY_max',
                  'accuracy_test','precision_test','recall_test','f1_test',
                  'accuracy_train','precision_train','recall_train','f1_train',
                  'params'
                 ]].to_json(f'ML_models/finalProbitModels/full_model_evals_{len(transforms1)}features.jsonl',orient='records',lines=True)

simple_model_evals[['growth_year','FY_max',
                  'accuracy_test','precision_test','recall_test','f1_test',
                  'accuracy_train','precision_train','recall_train','f1_train',
                  'params'
                 ]].to_json(f'ML_models/finalProbitModels/simple_model_evals_{len(transforms2)}features.jsonl',orient='records',lines=True)

current_model_evals[['growth_year','FY_max',
                  'accuracy_test','precision_test','recall_test','f1_test',
                  'accuracy_train','precision_train','recall_train','f1_train',
                  'params'
                 ]].to_json(f'ML_models/finalProbitModels/current_model_evals_{len(transforms_old)}features.jsonl',orient='records',lines=True)

all_model_evals[['growth_year','FY_max',
                  'accuracy_test','precision_test','recall_test','f1_test',
                  'accuracy_train','precision_train','recall_train','f1_train',
                  'params'
                 ]].to_json(f'ML_models/finalProbitModels/all_model_evals_{len(transforms_all)}features.jsonl',orient='records',lines=True)

pruned_model_evals[['growth_year','FY_max',
                  'accuracy_test','precision_test','recall_test','f1_test',
                  'accuracy_train','precision_train','recall_train','f1_train',
                  'params'
                 ]].to_json(f'ML_models/finalProbitModels/pruned_model_evals_variablefeatures.jsonl',orient='records',lines=True)


In [145]:
for k in transforms_all.keys():
    if k in pruned_model_evals[2]['params'].index:
        pass
    else:
        print(k)

L13
D1
D2
C2
S2
S7
S8
S10
S11
S12
S15
dS0
dS5
dS8
dS10
dS11
dS12
dS13
dS14
CS4
CS5
CS6
CS7
CS8
CS9
CS10
CS11
CS12
CS14
dCS2
dCS3
dCS4
dCS5
dCS7
dCS8
dCS9
dCS10
dCS11
dCS12
RS2
RS3
RS5
RS6
RS8
RS9
RS10
RS11
RS12
RS13
RS14
dRS0
dRS5
dRS7
dRS8
dRS9
dRS10
dRS11
dRS12
dRS13
dRS14
